In [2]:
import json
import os
import chromadb
import numpy as np
import torch
import shutil
import time
import gc
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import traceback

# ================= KONFIGURASI =================
INPUT_FILES = [
    r"D:\Penelitian supply chain\chunking\data_preprocessed_chunk_64.json"
]

MODELS_TO_RUN = [
    {"name": "BAAI/bge-m3", "short_name": "bge_m3"}
    
]

BASE_DB_PATH = "./chroma_db"
COLLECTION_NAME = "jurnal_supply_chain"
BATCH_SIZE = 16 
RESET_DB = True
# ===============================================

def force_cleanup(client_obj=None):
    if client_obj:
        del client_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    time.sleep(1)

def prepare_metadata(doc_item, chunk_idx):
    final_meta = {}
    if "metadata" in doc_item and isinstance(doc_item["metadata"], dict):
        source_meta = doc_item["metadata"]
    else:
        source_meta = {k: v for k, v in doc_item.items() if k not in ['id', 'chunks', 'metadata']}

    final_meta["original_doc_id"] = str(doc_item.get("id", "unknown"))
    final_meta["chunk_ke"] = str(chunk_idx)

    for key, value in source_meta.items():
        if isinstance(value, list):
            final_meta[key] = ", ".join([str(v) for v in value])
        elif value is None:
            final_meta[key] = ""
        else:
            final_meta[key] = str(value)
    return final_meta

def process_and_embed(model, model_short, file_path):
    chunk_size = file_path.split('_')[-1].replace('.json', '')
    persist_path = os.path.join(BASE_DB_PATH, model_short, f"db_{chunk_size}")
    
    print(f"\n{'='*60}")
    print(f"[RUNNING] Model: {model_short.upper()} | Chunk: {chunk_size}")
    print(f"[PATH] {persist_path}")
    print(f"{'='*60}")

    # --- STRATEGI BARU: SOFT RESET ---
    # Kita init client dulu, baru putuskan mau hapus folder atau hapus collection
    client = chromadb.PersistentClient(path=persist_path)
    
    if RESET_DB:
        try:
            # Coba hapus collection lama jika ada
            client.delete_collection(COLLECTION_NAME)
            print("[RESET] Collection lama berhasil dihapus (Soft Reset).")
        except ValueError:
            # Error muncul jika collection belum ada, abaikan saja
            pass
        except Exception as e:
            print(f"[WARN] Gagal reset collection: {e}")

    # Buat collection baru (atau get yang sudah kosong)
    collection = client.get_or_create_collection(name=COLLECTION_NAME)
    # ---------------------------------

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"[ERROR LOAD] Gagal membuka file JSON: {e}")
        return

    all_ids, all_docs, all_metas = [], [], []
    
    print("[PROCESS] Membaca data...")
    for doc in data:
        doc_id = str(doc.get("id", "unknown"))
        chunks = doc.get("chunks", [])
        
        for i, chunk_text in enumerate(chunks):
            if not isinstance(chunk_text, str) or not chunk_text.strip():
                continue
                
            chunk_idx = i + 1
            unique_id = f"{doc_id}_{chunk_idx}"
            all_ids.append(unique_id)
            all_docs.append(chunk_text)
            all_metas.append(prepare_metadata(doc, chunk_idx))

    total_docs = len(all_docs)
    print(f"[INFO] Total chunks valid: {total_docs}")
    
    if total_docs == 0:
        return

    try:
        for i in tqdm(range(0, total_docs, BATCH_SIZE), desc=f"Embedding {model_short}"):
            end_idx = i + BATCH_SIZE
            batch_docs = all_docs[i:end_idx]
            
            embeddings = model.encode(batch_docs, convert_to_tensor=True, normalize_embeddings=True)
            embeddings_list = embeddings.cpu().detach().tolist()

            collection.upsert(
                ids=all_ids[i:end_idx],
                embeddings=embeddings_list,
                documents=batch_docs,
                metadatas=all_metas[i:end_idx]
            )
        print(f"[DONE] Sukses tersimpan di: {persist_path}")
        
    except Exception as e:
        print(f"[ERROR EMBED] Error: {e}")
        traceback.print_exc()
    finally:
        # Cleanup resource
        del client
        force_cleanup() 

def main():
    # Hapus cache folder __pycache__ jika mengganggu (opsional)
    if os.path.exists("./__pycache__"):
        shutil.rmtree("./__pycache__", ignore_errors=True)

    for model_info in MODELS_TO_RUN:
        print(f"\n[INIT MODEL] Memuat {model_info['name']}...")
        try:
            model = SentenceTransformer(model_info['name'])
            
            for file_path in INPUT_FILES:
                if not os.path.exists(file_path):
                    continue
                process_and_embed(model, model_info["short_name"], file_path)
            
            del model
            force_cleanup()
            
        except Exception as e:
            print(f"[ERROR CRITICAL] {e}")
            traceback.print_exc()

if __name__ == "__main__":
    main()


[INIT MODEL] Memuat BAAI/bge-m3...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9cf0200b-b72c-47d4-a8ca-1a53fb81f127)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].



[RUNNING] Model: BGE_M3 | Chunk: 64
[PATH] ./chroma_db\bge_m3\db_64
[RESET] Collection lama berhasil dihapus (Soft Reset).
[PROCESS] Membaca data...
[INFO] Total chunks valid: 24925


Embedding bge_m3: 100%|██████████| 1558/1558 [2:30:05<00:00,  5.78s/it]  


[DONE] Sukses tersimpan di: ./chroma_db\bge_m3\db_64


In [ ]:
import chromadb
import numpy as np
import os

# --- KONFIGURASI (SAMAKAN DENGAN FILE UTAMA ANDA) ---
PERSIST_DIRECTORY = r"D:\Penelitian supply chain\embedding\chroma_db\bge_m3\db_64" 
COLLECTION_NAME = "jurnal_supply_chain"

def clean_outlier():
    print(f"Mengakses database di: {PERSIST_DIRECTORY}...")
    
    if not os.path.exists(PERSIST_DIRECTORY):
        print("Error: Direktori database tidak ditemukan!")
        return

    # 1. Connect ke DB
    client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)
    collection = client.get_collection(name=COLLECTION_NAME)
    
    # 2. Ambil semua data
    print("Mengambil seluruh data untuk analisis...")
    data = collection.get(include=['embeddings', 'documents'])
    
    embeddings = np.array(data['embeddings'])
    ids = data['ids']
    documents = data['documents']
    
    count = len(ids)
    print(f"Total dokumen: {count}")
    
    if count < 5:
        print("Data terlalu sedikit untuk mencari outlier.")
        return

    # 3. Hitung Outlier (Titik terjauh dari rata-rata)
    # Hitung titik tengah (centroid)
    centroid = np.mean(embeddings, axis=0)
    
    # Hitung jarak setiap titik ke centroid
    distances = np.linalg.norm(embeddings - centroid, axis=1)
    
    # Ambil index yang jaraknya paling jauh (Max Distance)
    outlier_idx = np.argmax(distances)
    
    outlier_id = ids[outlier_idx]
    outlier_text = documents[outlier_idx]
    outlier_dist = distances[outlier_idx]
    
    # 4. Tampilkan Hasil
    print("\n" + "="*50)
    print("!!! DITEMUKAN 1 OUTLIER EKSTRIM !!!")
    print("="*50)
    print(f"ID Dokumen : {outlier_id}")
    print(f"Jarak      : {outlier_dist:.4f} (Semakin besar semakin nyasar)")
    print("-" * 20)
    print("ISI TEKS (PREVIEW):")
    print(outlier_text[:500]) # Tampilkan 500 karakter pertama
    print("-" * 20)
    
    # 5. Konfirmasi Hapus
    ans = input("\nApakah Anda ingin MENGHAPUS data ini? (y/n): ").lower()
    
    if ans == 'y':
        collection.delete(ids=[outlier_id])
        print(f"SUKSES: Data dengan ID {outlier_id} telah dihapus permanen.")
        print("Silakan jalankan ulang visualisasi UMAP Anda.")
    else:
        print("Dibatalkan. Data tidak dihapus.")

if __name__ == "__main__":
    clean_outlier()

Mengakses database di: D:\Penelitian supply chain\embedding\chroma_db\bge_m3\db_64...
Mengambil seluruh data untuk analisis...
Total dokumen: 24925

!!! DITEMUKAN 1 OUTLIER EKSTRIM !!!
ID Dokumen : 1010_5
Jarak      : 0.9852 (Semakin besar semakin nyasar)
--------------------
ISI TEKS (PREVIEW):
the authors published by edp sciences roadef smai
--------------------
